<a href="https://colab.research.google.com/github/ProfAndersonVanin/IBD-016-Banco-de-Dados-N-o-Relacional/blob/main/aulas/AULA04/Aula04_BD_Nao_Relacional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nome do Banco de dados: aula04-bd
Dados de conexão:



```
r = redis.Redis(
    host='squirrel-shapely-dewy-15402.db.redis.io',
    port=17847,
    decode_responses=True,
    username="********",
    password="*****************",
)
```



# **1. INSTALAÇÃO E CONEXÃO COM O BANCO DE DADOS**

In [ ]:
# Instalar a biblioteca do Redis
!pip install redis

In [ ]:
# Conecte-se ao seu banco usando os dados anotados
import redis

r = redis.Redis(
    host='squirrel-shapely-dewy-15402.db.redis.io',
    port=17847,
    decode_responses=True,
    username="********",
    password="*****************",
)

print(r.ping())  # deve retornar True se a conexão funcionou

# **2. Primeiras operações**

## **2.1. Criar, consultar e remover uma chave**

In [ ]:
r.set("aula:mensagem", "Olá, Redis!")

In [ ]:
mensagem = r.get("aula:mensagem")
print(mensagem)

In [ ]:
r.delete("aula:mensagem")
print(r.get("aula:mensagem"))

> O prefixo `aula`: ajuda a identificar e agrupar as chaves criadas nesta atividade.



## **2.3. Verificar se uma chave existe**

In [ ]:
print(r.exists("aula:mensagem"))

> O resultado será `1` quando a chave existir e `0` quando não existir.



In [ ]:
for chave in r.scan_iter(match="aula:*"):
    print(chave)



> Em uma aplicação real, evite usar `KEYS *` em bases grandes, pois ele pode bloquear o servidor durante a varredura. Para inspeção, prefira **SCAN**.



# **3. Estruturas de dados do Redis**

## **3.1. Strings e contadores**
Strings podem guardar textos ou números. Contadores podem ser incrementados atomicamente pelo Redis.

In [ ]:
r.set("aula:visitas", 0)
r.incr("aula:visitas")
r.incrby("aula:visitas", 4)

print(r.get("aula:visitas"))

## **3.2. Hashes**
Hashes são úteis para representar campos de um objeto, como um aluno.

In [ ]:
aluno = {
    "nome": "Fulano da Silva",
    "curso": "Banco de Dados",
    "status": "ativo",
}

In [ ]:
r.hset("aula:aluno:1001", mapping=aluno)

In [ ]:
print(r.hget("aula:aluno:1001", "nome"))
print(r.hgetall("aula:aluno:1001"))

`r.hset("aula:aluno:1001", mapping=aluno)` ==> armazena os dados do dicionário aluno em um hash do Redis.

Supondo:

```
aluno = {
    "nome": "Ana Souza",
    "curso": "Banco de Dados",
    "status": "ativo",
}
```

O Redis criará a chave aula:aluno:1001 com estes campos:
```
aula:aluno:1001
├── nome   -> Ana Souza
├── curso  -> Banco de Dados
└── status -> ativo
```

**Partes do comando**

- `cliente.hset`: grava campos em um hash.
- `"aula:aluno:1001"`: nome da chave.
- `aula`: identifica a atividade;
- `aluno`: indica o tipo de dado;
- `1001` representa o identificador do aluno.
- `mapping=aluno`: informa que os campos e valores serão obtidos do dicionário Python.

É equivalente a escrever:

```
cliente.hset("aula:aluno:1001", "nome", "Ana Souza")
cliente.hset("aula:aluno:1001", "curso", "Banco de Dados")
cliente.hset("aula:aluno:1001", "status", "ativo")
```

Para consultar todos os campos:

`cliente.hgetall("aula:aluno:1001")`

Resultado:

````
{
    "nome": "Ana Souza",
    "curso": "Banco de Dados",
    "status": "ativo"
}
````


Atualização de um campo:

In [ ]:
r.hset("aula:aluno:1001", "status", "concluido")

In [ ]:
print(r.hget("aula:aluno:1001", "nome"))
print(r.hgetall("aula:aluno:1001"))

## **3.3. Listas**

Listas são adequadas para sequências e filas simples.

In [ ]:
r.delete("aula:fila")

In [ ]:
r.rpush("aula:fila", "atividade-1", "atividade-2", "atividade-3")

In [ ]:
proxima_atividade = r.lpop("aula:fila")

In [ ]:
print(proxima_atividade)
print(r.lrange("aula:fila", 0, -1))

`LPUSH` e `RPUSH` inserem elementos nas extremidades. `LPOP` e `RPOP` removem elementos.

## **3.4 Conjuntos**
Conjuntos não permitem valores repetidos e são úteis para tags ou grupos.

In [ ]:
r.sadd("aula:tecnologias", "redis", "python", "colab", "redis")

In [ ]:
print(r.smembers("aula:tecnologias"))

In [ ]:
print(r.sismember("aula:tecnologias", "java"))

In [ ]:
r.sadd("aula:tecnologias", "java")

In [ ]:
print(r.smembers("aula:tecnologias"))

## **3.5 Conjuntos ordenados**
Conjuntos ordenados associam cada membro a uma pontuação. São úteis para rankings.

In [ ]:
r.zadd(
    "aula:ranking",
    {
        "Ana": 95,
        "Bruno": 87,
        "Carla": 91,
    },
)

In [ ]:
ranking = r.zrevrange("aula:ranking", 0, -1, withscores=True)
print(ranking)

`zrevrange` retorna os maiores valores primeiro.

# **4. Expiração e cache**
Uma aplicação pode guardar um dado por tempo limitado. Isso é comum em caches, códigos temporários e sessões.

In [ ]:
r.setex("aula:codigo_temporario", 60, "ABC123")

In [ ]:
print(r.get("aula:codigo_temporario"))
print(r.ttl("aula:codigo_temporario"))

A chave expirará após 60 segundos. O método `ttl` retorna o tempo restante em segundos.

Também é possível definir a expiração depois:

In [ ]:
r.set("aula:cache", "resultado temporario")
r.expire("aula:cache", 120)

In [ ]:
print(r.get("aula:cache"))
print(r.ttl("aula:cache"))

# **5. Exemplo integrado: controle de acessos**

O exemplo abaixo combina hash, contador e expiração.

In [ ]:
usuario_id = "1001"
chave_usuario = f"aula:usuario:{usuario_id}"
chave_acessos = f"aula:acessos:{usuario_id}"

r.hset(
    chave_usuario,
    mapping={
        "nome": "Ana Souza",
        "perfil": "aluna",
    },
)

numero_de_acessos = r.incr(chave_acessos)
r.expire(chave_acessos, 3600)

print(r.hgetall(chave_usuario))
print(f"Acessos na última hora: {numero_de_acessos}")

O uso de `INCR` é seguro para incrementos concorrentes: o Redis executa a operação de forma atômica.

# **6. Limpeza da atividade**
Ao final, remova somente as chaves usadas pela aula:

In [ ]:
chaves_da_aula = list(r.scan_iter(match="aula:*") )

if chaves_da_aula:
    r.delete(*chaves_da_aula)

print("Chaves removidas:", len(chaves_da_aula))